In [2]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
import accelerate
import transformers

In [3]:
data = pd.read_csv("/content/data_finbert.csv")

In [4]:
data

,Sentence,Sentiment
0,apple breaks major support here are some level...,-1
1,apple up almost 20 from its february lows with...,1
2,fintech provider cafn cachet financial solutio...,1
3,fusioniq new positive timing signal on sbux to...,1
4,longpos tsla 256 break out thru 50 200 dma 197...,1
...,...,...
17732,what stocks large players are selling http stk...,-1
17733,what is up with hk from jan to now it s been a...,-1
17734,wins 98 acceptance 23 december 2009 finnish in...,0
17735,countryelements designed by patricia burt this...,0


In [5]:
data = data.rename(columns={
    "Sentence": "text",
    "Sentiment": "label"
})
label_map = {
    -1: 0,
    0: 1,
    1: 2
}

data["label"] = data["label"].map(label_map)

In [6]:
data

,text,label
0,apple breaks major support here are some level...,0
1,apple up almost 20 from its february lows with...,2
2,fintech provider cafn cachet financial solutio...,2
3,fusioniq new positive timing signal on sbux to...,2
4,longpos tsla 256 break out thru 50 200 dma 197...,2
...,...,...
17732,what stocks large players are selling http stk...,0
17733,what is up with hk from jan to now it s been a...,0
17734,wins 98 acceptance 23 december 2009 finnish in...,1
17735,countryelements designed by patricia burt this...,1


In [7]:
train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    stratify=data["label"],
    random_state=42
)


In [10]:
train_data

,text,label
11035,pvr reports q3 profit at rs 31 55 crore,1
7614,indian cashew exports to touch new high this f...,2
14705,the board of directors proposes to the shareho...,1
4024,dlf surges on reports to launch cmbs issue in ...,2
978,according to ceo hannu syrj ñnen a new common ...,1
...,...,...
969,according to a rehu s managing director jouko ...,2
2282,barclays appoints heads of equities for india ...,1
17181,whitbread boss andy harrison defends sales fal...,0
7765,inha works has invested in its product develop...,1


In [11]:
train_dataset = Dataset.from_pandas(train_data)
test_dataset = Dataset.from_pandas(test_data)

In [12]:
tokenizer = AutoTokenizer.from_pretrained(
    "ProsusAI/finbert"
)

In [13]:
def tokenize_function(example):

    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    tokens["labels"] = example["label"]

    return tokens

In [14]:
train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

Map:   0%|          | 0/14189 [00:00<?, ? examples/s]

Map:   0%|          | 0/3548 [00:00<?, ? examples/s]

In [15]:
training_args = TrainingArguments(
    output_dir="./finbert_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False
)

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

In [18]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.620456,0.422714
2,0.326063,0.468262
3,0.205318,0.552700
4,0.136725,0.685440
5,0.107367,0.733032


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=4435, training_loss=0.26300346434721145, metrics={'train_runtime': 1901.0492, 'train_samples_per_second': 37.319, 'train_steps_per_second': 2.333, 'total_flos': 4666645355178240.0, 'train_loss': 0.26300346434721145, 'epoch': 5.0})

In [9]:
import torch
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

# -----------------------------------
# LOAD PRETRAINED FINBERT
# (WITHOUT FINE-TUNING)
# -----------------------------------

tokenizer = AutoTokenizer.from_pretrained(
    "ProsusAI/finbert"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert"
)

# Set evaluation mode
model.eval()

# -----------------------------------
# LOSS FUNCTION
# SAME AS TRAINING
# -----------------------------------

criterion = CrossEntropyLoss()

total_loss = 0

# -----------------------------------
# LOOP THROUGH DATA
# -----------------------------------

for _, row in test_data.iterrows():

    text = row["text"]
    label = row["label"]

    # Tokenize input text
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    # Convert label to tensor
    labels = torch.tensor([label])

    # Disable gradient computation
    with torch.no_grad():

        outputs = model(**inputs)

        logits = outputs.logits

        # Compute loss
        loss = criterion(logits, labels)

    total_loss += loss.item()

# -----------------------------------
# AVERAGE LOSS
# -----------------------------------

baseline_loss = total_loss / len(data)

print(f"\nBaseline FinBERT Loss: {baseline_loss:.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Baseline FinBERT Loss: 0.5666


In [49]:
ans[0]

'neutral'

In [20]:
!pip uninstall -y torchao
!pip install -U torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.0 MB/s eta 0:00:00


In [21]:
# =========================================================
# PEFT / LoRA Fine-Tuning on FinBERT
# =========================================================

# Install:
# pip install peft transformers accelerate datasets

import pandas as pd
import torch

from sklearn.model_selection import train_test_split
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

# =========================================================
# DATA
# =========================================================

# Your dataframe already exists:
# data

# Columns:
# Text
# label

# Rename columns for HuggingFace compatibility
data = data.rename(columns={
    "Text": "text"
})

# =========================================================
# TRAIN TEST SPLIT
# =========================================================

train_df, test_df = train_test_split(
    data,
    test_size=0.2,
    stratify=data["label"],
    random_state=42
)

# =========================================================
# CONVERT TO DATASET
# =========================================================

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

# =========================================================
# TOKENIZER
# =========================================================

tokenizer = AutoTokenizer.from_pretrained(
    "ProsusAI/finbert"
)

# =========================================================
# TOKENIZATION FUNCTION
# =========================================================

def tokenize_function(example):

    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # Trainer requires "labels"
    tokens["labels"] = example["label"]

    return tokens

# Tokenize
train_dataset = train_dataset.map(tokenize_function)
test_dataset = test_dataset.map(tokenize_function)

# Remove unnecessary columns
train_dataset = train_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

# Torch format
train_dataset.set_format("torch")
test_dataset.set_format("torch")

# =========================================================
# LOAD PRETRAINED FINBERT
# =========================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "ProsusAI/finbert",
    num_labels=3
)

# =========================================================
# LORA CONFIGURATION
# =========================================================

lora_config = LoraConfig(

    task_type=TaskType.SEQ_CLS,

    r=8,                  # Rank

    lora_alpha=16,

    lora_dropout=0.1,

    bias="none"
)

# =========================================================
# APPLY PEFT / LORA
# =========================================================

model = get_peft_model(
    model,
    lora_config
)

# Show trainable params
model.print_trainable_parameters()

# =========================================================
# TRAINING ARGUMENTS
# =========================================================

training_args = TrainingArguments(

    output_dir="./finbert_peft_results",

    eval_strategy="epoch",

    save_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=16,

    per_device_eval_batch_size=16,

    num_train_epochs=5,

    weight_decay=0.01,

    logging_dir="./logs",

    logging_steps=50,

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False
)

# =========================================================
# TRAINER
# =========================================================

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

# =========================================================
# TRAIN PEFT MODEL
# =========================================================

trainer.train()

Map:   0%|          | 0/14189 [00:00<?, ? examples/s]

Map:   0%|          | 0/3548 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


trainable params: 297,219 || all params: 109,781,766 || trainable%: 0.2707


Epoch,Training Loss,Validation Loss
1,0.628259,0.620314
2,0.590205,0.584283
3,0.594059,0.568258
4,0.575523,0.562879
5,0.565355,0.559682


TrainOutput(global_step=4435, training_loss=0.6778684807468831, metrics={'train_runtime': 1288.5866, 'train_samples_per_second': 55.056, 'train_steps_per_second': 3.442, 'total_flos': 4682839558279680.0, 'train_loss': 0.6778684807468831, 'epoch': 5.0})